# Sudoku RL : suivre l'apprentissage

Chaque entraînement de `06_reinforce.py` écrit une ligne par itération dans `logs/*.csv` : `retour`, `resolues`, `coups`, `perte`, `entropie`.

Une itération = 64 parties jouées puis un pas d'apprentissage. Chaque point est donc **bruité** (64 parties seulement) : on lisse avec une moyenne glissante pour voir la tendance.

Le chiffre à suivre est le **nombre de coups par partie** : plus il est proche du minimum (le nombre de cases vides), meilleur est le réseau.

In [ ]:
from pathlib import Path
import csv
import matplotlib.pyplot as plt

LOGS = next((p for p in (Path("..") / "logs", Path("logs")) if p.exists()), None)
if LOGS is None:
    raise FileNotFoundError("dossier logs introuvable : ouvrir le notebook depuis le dépôt mini-llm")
# 30 cases vides : minimum 30 coups. Grilles complètes : minimum environ 50.
ESSAIS_30 = {
    "GAMMA 0,99 (essai 7, 2000 it.)": "sudoku_reinforce_v30_lr3e3.csv",
    "GAMMA 0 graine 0": "sudoku_reinforce_v30_lr3e3_g0.csv",
    "GAMMA 0 graine 1": "sudoku_reinforce_v30_lr3e3_g0_graine1.csv",
    "GAMMA 0 graine 2": "sudoku_reinforce_v30_lr3e3_g0_graine2.csv",
    "GAMMA 0 graine 3": "sudoku_reinforce_v30_lr3e3_g0_graine3.csv",
}
ESSAIS_COMPLET = {
    "lot 64 (essai 10)": "sudoku_reinforce_complet_lr3e3_g0.csv",
    "lot 256 (essai 11)": "sudoku_reinforce_complet_lot256.csv",
    "lr décroissant (essai 12)": "sudoku_reinforce_complet_lrdecroit.csv",
    "2000 it. (essai 13)": "sudoku_reinforce_complet_2000it.csv",
    "voit les refus (essai 14)": "sudoku_reinforce_complet_refus.csv",
    "refus + entropie 0,02 (essai 15)": "sudoku_reinforce_complet_refus_entropie.csv",
    "référence graine 0": "sudoku_reference_g0.csv",
    "référence graine 1": "sudoku_reference_g1.csv",
    "référence graine 2": "sudoku_reference_g2.csv",
    "référence graine 3": "sudoku_reference_g3.csv",
}

def lire(nom_fichier, colonne):
    chemin = LOGS / nom_fichier
    if not chemin.exists():
        return [], []
    with open(chemin) as f:
        lignes = list(csv.DictReader(f))
    return [int(l["iteration"]) for l in lignes], [float(l[colonne]) for l in lignes]

def lisser(valeurs, fenetre=20):
    """Moyenne glissante : chaque point = moyenne des `fenetre` dernières itérations."""
    sortie = []
    for i in range(len(valeurs)):
        morceau = valeurs[max(0, i - fenetre + 1): i + 1]
        sortie.append(sum(morceau) / len(morceau))
    return sortie

def tracer(essais, colonne, titre, minimum=None, fenetre=20):
    for nom, fichier in essais.items():
        x, y = lire(fichier, colonne)
        if not y:  # essai absent ou pas encore terminé
            print(f"(rien à tracer pour « {nom} » : {fichier} absent ou vide)")
            continue
        plt.plot(x, lisser(y, fenetre), label=f"{nom} : fin {sum(y[-50:]) / len(y[-50:]):.2f}")
    if minimum is not None:
        plt.axhline(minimum, color="black", linestyle=":", label=f"minimum possible ({minimum})")
    plt.xlabel("itération")
    plt.ylabel(titre)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## Nombre de coups par partie (30 cases vides)

Avant apprentissage : environ 36 coups. Minimum : 30. « fin » = moyenne des 50 dernières itérations.

In [ ]:
tracer(ESSAIS_30, "coups", "coups par partie (moyenne glissante)", minimum=30)

## Entropie : le réseau se décide-t-il ?

L'entropie mesure l'hésitation du réseau entre les coups autorisés : haute au départ (≈ 2,8, quasi hasard), basse quand il est sûr de lui. Elle baisse plus vite avec GAMMA 0.

In [ ]:
tracer(ESSAIS_30, "entropie", "entropie (moyenne glissante)")

## Grilles complètes (environ 50 cases vides)

Le vrai problème. Essais 10, 11 et 12 (terminés). Si la courbe est vide : exécute d'abord la cellule de définitions (la première), puis celle-ci.

In [ ]:
tracer(ESSAIS_COMPLET, "coups", "coups par partie (moyenne glissante)", minimum=50)

### L'essai 14 diverge : l'entropie s'effondre

Le réseau qui voit les refus (essai 14) devient trop sûr de lui : son entropie tombe à 0, il rejoue le même coup faux 150 fois et ne peut plus apprendre (voir `docs/suivi_sudoku_rl.md`, « Étude de cas »). La courbe d'entropie le montre bien avant la fin.

In [ ]:
tracer(ESSAIS_COMPLET, "entropie", "entropie (moyenne glissante)")

**Une entropie basse n'est pas un bon signe en soi** : elle veut dire « le réseau est sûr de lui », même s'il a tort. Il faut la lire avec la part de grilles résolues : pour l'essai 14, l'entropie à 0 s'accompagne de 0 % de grilles résolues.

In [ ]:
tracer(ESSAIS_COMPLET, "resolues", "part de grilles résolues (moyenne glissante)")

**À essayer** : change `fenetre=20` en `fenetre=5` (courbe plus nerveuse) ou `fenetre=80` (tendance plus nette), et relance seulement cette cellule. Ajoute un nouvel essai dans `ESSAIS_30` : il apparaît sur toutes les courbes sans rien recalculer.

Rappel : une seule graine peut tromper. Quatre courbes GAMMA 0 qui se ressemblent valent mieux qu'une.